Inputs: Training Set (Data, Labels)

Outputs: Metrics: Loss and Accuracy

## Setup

In [ ]:
# sync with github so my imports are here
!git clone https://github.com/ellylai/10707-project.git
%cd 10707-project

!pip install -q transformers datasets scikit-learn

import sys
sys.path.append("/content/10707-project")

In [2]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# Adding specific directories to sys.path since they are not recognized as Python packages (missing __init__.py files).
# The parent directory '/content/10707-project' was already added in a previous cell.
import sys
if "/content/10707-project/datasets" not in sys.path:
    sys.path.append("/content/10707-project/datasets")
if "/content/10707-project/utils" not in sys.path:
    sys.path.append("/content/10707-project/utils")

# Now import modules directly by their filenames
from download_datasets import *
from training_utils import *
from create_dataset_splits import *
from get_waveform_dataset import *

## Unzip & Upload `.wav` Files

In [ ]:
# 1. Mount Drive to access the zips
from google.colab import drive
import os

# 1. Mount your Drive
drive.mount('/content/drive')

# 2. Path to the shortcut you just created
# (Replace 'XMAD-dataset' with whatever you named the shortcut)
!ls /content/drive/MyDrive/
dataset_path = '/content/drive/MyDrive/XMAD-dataset'

# 3. Verify you can see the zips
print("Files in shared folder:")
!ls "{dataset_path}"

!mkdir -p /content/xmad_local
# languages = ['en.zip', 'zh-cn.zip', 'es.zip'] # Add more as needed: 'ar.zip', 'de.zip', etc.
languages = ['ar.zip', 'de.zip', 'ro.zip', 'ru.zip'] + ['en.zip', 'zh-cn.zip', 'es.zip']

for lang in languages:
    zip_path = os.path.join(dataset_path, lang)
    if os.path.exists(zip_path):
        print(f"Unzipping {lang}...")
        # -n skips files that already exist to save time if you re-run
        !unzip -nq "{zip_path}" -d /content/xmad_local/
    else:
        print(f"Warning: {lang} not found in {dataset_path}")

print("Unzip process complete.")

In [ ]:
!pip install awscli
!aws s3api get-bucket-location --bucket 10707-project

In [ ]:
# Sync to S3
# 1. Increase the number of parallel threads (Default is 10, set to 50+)
!aws configure set default.s3.max_concurrent_requests 100

# 2. Lower the threshold for multipart uploads (Good for small-medium files)
!aws configure set default.s3.multipart_threshold 8MB

# 3. Now run the sync again
!aws s3 sync /content/xmad_local/ s3://10707-project/xmad_bench/ --size-only --only-show-errors

## Creating the Splits `.csv` File & Uploading to S3

In [ ]:
import pandas as pd
import os
from pathlib import Path

# Root directory where your languages are unzipped
root_dir = "/content/xmad_local"
all_data = []

# Walk through all directories to find meta.csv files
for path in Path(root_dir).rglob('meta.csv'):
    # Read the individual metadata file
    df = pd.read_csv(path)

    # Identify the dataset source from the folder structure
    parent_folder = path.parent.name

    if "commonvoice" in parent_folder.lower():
        # CommonVoice contains the internal 'train' and 'test' labels
        # We use 'train' for training and 'test' as our Validation set
        df['split'] = df['split'].map({'train': 'train', 'test': 'val'})
    else:
        # AISHELL-3, M-AILABS, etc., are strictly for Cross-Domain Testing
        df['split'] = 'test'

    # Convert relative 'sample_name' paths to absolute 'file' paths for the DataLoader
    # This ensures your training_pipeline.ipynb can find the .wav files
    # The original error was 'KeyError: 'file'' because the column is actually 'sample_name'
    df['file'] = df.apply(
        lambda row: os.path.join(
            path.parent,
            'fake' if row['is_fake'] == 1 else 'real',
            str(row['sample_name'])
        ),
        axis=1
    )

    # 3. Rename label for clarity in your DataLoader
    df['label'] = df['is_fake']

    all_data.append(df)

# Combine all parsed metadata into one master dataframe
if all_data:
    master_df = pd.concat(all_data, ignore_index=True)

    # Ensure the data directory exists in your cloned repo
    os.makedirs("/content/10707-project/data", exist_ok=True)

    # Save the manifest to the path expected by your notebook
    output_path = "/content/10707-project/data/speechfake_splits.csv"
    master_df.to_csv(output_path, index=False)

    print(f"Successfully created: {output_path}")

    # upload to s3 bucket
    s3_dest = "s3://10707-project/xmad_bench/metadata/speechfake_splits.csv"
    !aws s3 cp {output_path} {s3_dest}
    print(f"Successfully uploaded to S3: {s3_dest}")

    # double checking
    print("Split Distribution:")
    print(master_df['split'].value_counts())

    print("Unique values in the split column:")
    print(master_df['split'].unique())

    print("\nRows where split is NaN:")
    print(master_df['split'].isna().sum())
else:
    print("No meta.csv files found. Ensure the unzip process finished correctly.")

In [ ]:
# FIX ONLY, should not need to run if we download from S3
import pandas as pd

# 1. Load the manifest you just created
file_path = "/content/10707-project/data/speechfake_splits.csv"
master_df = pd.read_csv(file_path)

# 2. Fix the NaNs: Anything that failed to map is almost certainly the
# 'test' portion of CommonVoice that we want to use as 'val'
master_df['split'] = master_df['split'].fillna('val')

# 3. Double check the distribution now
print("Fixed Split Distribution:")
print(master_df['split'].value_counts())

# 4. Save it back so the training cell sees the fix
master_df.to_csv(file_path, index=False)

# 5. Sync the fixed version to S3 so your EC2 gets the right one later
!aws s3 cp {file_path} s3://10707-project/xmad_bench/metadata/speechfake_splits.csv

## To Download Directly from AWS S3 Bucket, including `.csv`

In [ ]:
# to download from AWS S3
# 1. Install AWS CLI (Not pre-installed on Colab)
!pip install -q awscli

# 2. OPTIONAL: If you saved your credentials in a file on Drive, load them here
# Otherwise, run !aws configure to log in again

# 3. Optimize for high-speed download
!aws configure set default.s3.max_concurrent_requests 100
!aws configure set default.s3.multipart_threshold 8MB

# 4. Sync from S3 to Local Colab Disk
# We use --size-only to skip unnecessary checks and speed up the process
import os
os.makedirs("/content/xmad_local", exist_ok=True)

print("Starting high-speed download from S3...")
!aws s3 sync s3://10707-project/xmad_bench/ /content/xmad_local/ --size-only --only-show-errors

print("Download complete. Checking local storage:")
!df -h /content

## Dataset & Dataloader

In [40]:
import torch
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class XMADDataset(Dataset):
    def __init__(self, df, target_sr=16000, max_seconds=4.0):
        self.df = df
        self.target_sr = target_sr
        self.max_samples = int(target_sr * max_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file']
        label = row['is_fake'] # Corrected: Use 'is_fake' column directly

        # Load audio
        waveform, sr = torchaudio.load(file_path)

        # Resample if necessary
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Pad or Trim to fixed length (crucial for batching)
        if waveform.shape[1] > self.max_samples:
            waveform = waveform[:, :self.max_samples]
        else:
            padding = self.max_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))

        return waveform.squeeze(0), torch.tensor(label)

In [ ]:
# Load your master manifest
master_df = pd.read_csv("/content/10707-project/data/speechfake_splits.csv")

# Filter dataframes by split
train_df = master_df[master_df['split'] == 'train'].reset_index(drop=True)
val_df = master_df[master_df['split'] == 'val'].reset_index(drop=True)
test_df = master_df[master_df['split'] == 'test'].reset_index(drop=True)

# VERIFICATION ABOUT THE SPLIT RATIOS
def print_stratification(df, name):
    print(f"\n--- {name} Stratification ---")
    print(f"Total Samples: {len(df)}")

    for col in ['is_fake', 'label', 'meta']:
        if col in df.columns:
            print(f"\nProportions for '{col}':")
            # normalize=True gives ratios (0.0 to 1.0)
            print(df[col].value_counts(normalize=True).map(lambda n: f'{n:.2%}'))
        else:
            print(f"\nColumn '{col}' not found in this split.")

# Run the check
print_stratification(train_df, "Train")
print_stratification(val_df, "Validation")
print_stratification(test_df, "Test")

# Create Dataset objects
train_dataset = XMADDataset(train_df)
val_dataset = XMADDataset(val_df)
test_dataset = XMADDataset(test_df)

# Create DataLoaders
# Set num_workers to 2 or 4 to speed up loading on Colab
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"Loaders created: {len(train_loader)} train batches, {len(val_loader)} val batches, {len(test_loader)} test batches.")

## Start Training

In [12]:
# args/configs
d_args = {
    "filts": [[1, 32], [32, 32], [32, 64], [64, 64], [64, 128]], # Example AASIST filter bank
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5],
    "temperatures": [2.0, 2.0, 1.0],
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8  # Keep small for W2V2
lr = 0.0001
epochs = 10

In [ ]:
# wandb and huggingface setup
import wandb
from huggingface_hub import HfApi, login

# Initialize W&B
wandb.init(
    project="audio-deepfake-detection",
    config={
        "learning_rate": lr,
        "architecture": "W2V2_AASIST",
        "dataset": "SpeechFake",
        "epochs": epochs,
    }
)

In [ ]:
# import model and initialize optimizer
import sys
# Ensure the baseline directory is in sys.path for direct module import
if "/content/10707-project/baseline" not in sys.path:
    sys.path.append("/content/10707-project/baseline")

from w2v2_aasist import W2V2_AASIST

model = W2V2_AASIST(d_args).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [ ]:
# HuggingFace Setup & Verification

import os
import huggingface_hub

# 1. Force the environment variable to your new token
# This overrides the previous secret for the rest of this session
os.environ["HF_TOKEN"] = "..."

# 2. Re-initialize the API object
# This forces the client to read the new environment variable
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])

# 3. Verify
try:
    user_info = api.whoami()
    print(f"Successfully switched! Logged in as: {user_info['name']}")
    print(f"Can write to Joel-10707-Project-S26: {'Yes' if any(org['name'] == 'Joel-10707-Project-S26' for org in user_info['orgs']) else 'No'}")
except Exception as e:
    print(f"Switch failed: {e}")

# Test the connection before training
repo_id = "Joel-10707-Project-S26/baseline-w2v2aasist"

# 1. Create a tiny test file locally
test_file = "elly_write_test.txt"
with open(test_file, "w") as f:
    f.write("Elly has write access and is ready for 10707 training!")

# 2. Attempt to upload DIRECTLY (No create_repo call)
try:
    print(f"Testing direct write access to {repo_id}...")
    api.upload_file(
        path_or_fileobj=test_file,
        path_in_repo="tests/elly_write_test.txt",
        repo_id=repo_id,
        # Since it's a private repo, specify the repo_type if needed,
        # though it defaults to model
        repo_type="model"
    )
    print("\n✅ Success! You can push to the repo.")
except Exception as e:
    print(f"\n❌ Still failing: {e}")

In [ ]:
# TRAIN
for epoch in range(epochs):
    # --- Existing Training Logic ---
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Val Acc {val_acc:.4f}")

    # --- Checkpointing & HF Upload ---
    checkpoint_name = f"checkpoint-epoch-{epoch+1}.pt"
    checkpoint_path = os.path.join("/content/", checkpoint_name)

    # Save locally first
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        'val_acc': val_acc,
    }, checkpoint_path)

    # Upload to Hugging Face
    print(f"Uploading {checkpoint_name} to Hugging Face...")
    api.upload_file(
        path_or_fileobj=checkpoint_path,
        path_in_repo=checkpoint_name,
        repo_id=repo_id,
        commit_message=f"Upload checkpoint for epoch {epoch+1}"
    )

    # Optional: Delete local file to save Colab disk space
    os.remove(checkpoint_path)